# 🌟 Twinkle Eval｜選擇題評測教學

## 專案資訊

| 項目 | 內容 |
|------|------|
| GitHub | https://github.com/ai-twinkle/Eval |
| PyPI | `twinkle-eval` |
| 授權 | MIT |
| Python | ≥ 3.11 |
| Colab 工作目錄 | `/content/Eval/` |

---

## 開場白

2025 年初，推理模型（reasoning model）開始大量出現。這類模型在給出答案之前，會先輸出大量的思考過程（chain-of-thought），導致每次 API 呼叫的等待時間遠高於傳統模型。

傳統評測框架採用**同步、逐題呼叫**的設計——一題答完才問下一題——面對推理模型時，整個 benchmark 動輒耗費數小時，白白浪費 GPU 閒置時間。

**Twinkle Eval** 因此而生：以**並行 API 請求**為核心，讓每道題目同時送出，實測相比傳統框架快 **9–17 倍**。除了速度，Twinkle Eval 透過**選項隨機排列**消除模型的位置偏好（position bias），讓評測結果更客觀。

## 本 Notebook 範圍

本 Notebook 專注於**選擇題**的三種評測方法：

| 方法 | 原理簡述 | 最適情境 |
|------|---------|--------|
| **Pattern** | 正規表達式比對文字 | 通用首選、指令遵循強的模型 |
| **Box** | 提取 `\boxed{}` 內容 | 推理模型 |
| **Logit** | 比較各選項 log-probability | 多選項題、不依賴格式 |

> 📓 數學開放題（GSM8K、AIME 等）的評測方法請見 **`02_數學評測.ipynb`**。

---

# 第一章：環境安裝

選擇題評測只需要安裝基本套件，不需要數學擴充。若未來同時要跑數學評測，請改安裝 `twinkle-eval[math]`（詳見 `02_數學評測.ipynb`）。

In [ ]:
!pip install -q twinkle-eval

In [ ]:
import twinkle_eval
from twinkle_eval.metrics import get_available_methods

info = twinkle_eval.get_info()
print(f"套件版本：{info['version']}")
print(f"可用評測方法：{get_available_methods()}")

---

# 第二章：準備資料集

## 選擇題的欄位格式

每筆資料需要包含題目、各選項與正確答案：

```json
{
  "question": "題目文字",
  "A": "選項 A",
  "B": "選項 B",
  "C": "選項 C",
  "D": "選項 D",
  "answer": "A"
}
```

> 💡 選項數量不限於 A–D，Twinkle Eval 會動態偵測所有大寫字母欄位，因此 MMLU-Pro 這類有 10 個選項（A–J）的資料集也完整支援。

我們直接 clone 範例資料集來使用。

In [ ]:
import os

if not os.path.exists("/content/Eval"):
    !git clone -q https://github.com/ai-twinkle/Eval.git /content/Eval

os.chdir("/content/Eval")
print(f"工作目錄：{os.getcwd()}")

In [ ]:
import json

# 本 Notebook 只使用選擇題資料集
mcq_datasets = {
    "TMMLU+（繁體中文，4 選項）":   "datasets/example/tmmluplus/test.jsonl",
    "MMLU-Pro（英文，10 選項 A–J）": "datasets/example/mmlu_pro/test.jsonl",
}

for name, path in mcq_datasets.items():
    with open(path) as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    option_keys = [k for k in first if k.isupper() and len(k) == 1]
    print(f"📂 {name}")
    print(f"   筆數：{len(lines)}　選項欄位：{option_keys}")
    print(f"   範例：{first['question'][:50]}...")
    print()

---

# 第三章：設定檔（config.yaml）

Twinkle Eval 以 YAML 設定檔驅動所有行為。一份設定檔包含三個必填區塊：

```yaml
llm_api:        # API 端點、金鑰、速率限制
model:          # 模型名稱與生成參數
evaluation:     # 資料集路徑與評測方法
```

其中 `datasets_prompt_map` 可以針對特定資料集指定使用哪種語言的 system prompt（`zh` 或 `en`）。

> ⚠️ **請將下方的 API 資訊替換為你實際的端點與金鑰。**

In [ ]:
API_BASE_URL = "https://your-api-endpoint/v1"
API_KEY      = "sk-your-api-key-here"
MODEL_NAME   = "your-model-name"

In [ ]:
import yaml

base_config = {
    "llm_api": {
        "base_url": API_BASE_URL,
        "api_key": API_KEY,
        "api_rate_limit": -1,
        "max_retries": 3,
        "timeout": 60,
    },
    "model": {
        "name": MODEL_NAME,
        "temperature": 0.0,
        "max_tokens": 512,
    },
    "logging": {"level": "INFO"},
}

mcq_evaluation = {
    "dataset_paths": [
        "datasets/example/tmmluplus/",
        "datasets/example/mmlu_pro/",
    ],
    "datasets_prompt_map": {
        "datasets/example/mmlu_pro/": "en",
    },
}

configs = {
    "config_mcq_pattern.yaml": {**base_config, "evaluation": {**mcq_evaluation, "evaluation_method": "pattern"}},
    "config_mcq_box.yaml":     {**base_config, "evaluation": {**mcq_evaluation, "evaluation_method": "box"}},
    "config_mcq_logit.yaml":   {**base_config, "evaluation": {**mcq_evaluation, "evaluation_method": "logit"}},
}

for filename, cfg in configs.items():
    with open(filename, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)
    print(f"✅ 已建立 {filename}")

---

# 第四章：Pattern 評測

## 原理

Pattern 評測是最直觀的方式：在模型的**文字回覆**中，用一系列正規表達式搜尋答案字母。Twinkle Eval 內建數十種中英文常見格式：

| 模型輸出 | 匹配到 |
|---------|-------|
| `答案：B` | B |
| `正確答案為 **C**` | C |
| `The correct answer is:\nA.` | A |
| `答案應該是: 選項 D` | D |

Pattern 依賴模型「按照格式說出答案」，對**指令遵循能力強**的模型效果很好，是最常用的首選方法。若模型習慣把答案包在 `\boxed{}` 裡，會有較高的無法解析率，此時應改用 Box 評測。

In [ ]:
!python -m twinkle_eval.cli --config config_mcq_pattern.yaml

---

# 第五章：Box 評測

## 原理

許多推理模型習慣將最終答案包在 LaTeX 的 `\boxed{}` 語法中。Box 評測專門提取這種格式：

```
模型輸出：「根據以上分析，正確答案是 $$\boxed{B}$$」
提取結果：B
```

支援 `\boxed{X}` 和 `\box{X}`，並以括號計數法處理巢狀大括號。

## 無法解析（Unparsed）

若模型回覆中完全沒有 `\boxed{}`，該題會被標記為**無法解析**，計為答錯，並在評測結束時顯示：

```
⚠️  無法解析: 4/20 (20.0%)
```

無法解析率偏高時，可調整 system prompt 要求模型以 `\boxed{}` 框住答案，或改用 Pattern 評測。

In [ ]:
!python -m twinkle_eval.cli --config config_mcq_box.yaml

---

# 第六章：Logit 評測

## 原理

Pattern 和 Box 都屬於**生成式路徑**——讓模型「說出答案」再解析文字。Logit 走完全不同的路：**直接問模型「哪個選項機率最高」**，不產生任何文字輸出。

做法是利用 `/v1/completions` 的 **log probability** API，對每個選項各送一次請求，取 log-prob 最大的：

```
P(" A" | context) = -0.2  ← 最高
P(" B" | context) = -1.5
P(" C" | context) = -2.1
P(" D" | context) = -3.0
預測答案 → A
```

## 為何 Logit 適合多選項題

MMLU-Pro 有 10 個選項（A–J）。Pattern 評測若正規表達式未完整涵蓋 E–J，就會漏掉答案。Logit 路徑完全繞過文字解析，對任意數量的選項一視同仁，因此在 MMLU-Pro 這類題目上表現通常更好。

## Logit 只適合選擇題

Logit 路徑的前提是「題目有離散的選項可以比較機率」。數學開放題沒有選項，Logit 無從計算，**強行搭配數學資料集會得到 0 分**。若需要評測數學題，請見 `02_數學評測.ipynb`。

In [ ]:
!python -m twinkle_eval.cli --config config_mcq_logit.yaml

---

# 第七章：三種方法比較

跑完三種評測後，我們讀取結果來做比較。

In [ ]:
import glob, json

def load_results(n=3):
    files = sorted(glob.glob("results/results_*.json"))[-n:]
    results = []
    for f in files:
        with open(f) as fp:
            data = json.load(fp)
        method = data.get("config", {}).get("evaluation", {}).get("evaluation_method", "?")
        results.append((method.upper(), data["dataset_results"]))
    return results

all_results = load_results(3)
methods = [r[0] for r in all_results]
ds_order = ["tmmluplus", "mmlu_pro"]

print(f"{'資料集':<16}" + "".join(f"{m:>14}" for m in methods))
print("-" * (16 + 14 * len(methods)))

for ds_key in ds_order:
    row = f"{ds_key:<16}"
    for method, ds_results in all_results:
        matched = next((v for k, v in ds_results.items() if ds_key in k), None)
        if matched:
            acc = matched.get("average_accuracy", 0) * 100
            uparsed = matched.get("average_unparsed_rate", 0) * 100
            cell = f"{acc:.1f}%" + (" ⚠" if uparsed > 0 else "")
        else:
            cell = "N/A"
        row += f"{cell:>14}"
    print(row)

print("\n⚠ = 有無法解析的題目（predicted=None）")

## 特性對照

| | Pattern | Box | Logit |
|---|---|---|---|
| **原理** | 正規表達式比對 | 提取 `\boxed{}` | 比較 log-probability |
| **依賴格式** | 是 | 是（需要 `\boxed{}`）| **否** |
| **API 呼叫數** | 1 次 / 題 | 1 次 / 題 | N 次 / 題（N = 選項數）|
| **無法解析風險** | 有 | 有 | **無** |
| **多選項（A–J）** | 可能漏字母 | 可能漏字母 | **完整支援** |
| **最適情境** | 通用、指令遵循強 | 推理模型 | 多選項、公平比較 |

## 選擇建議

- **一般用途** → 先用 **Pattern**，若無法解析率 > 10% 改用 Box
- **推理模型**（DeepSeek-R1、QwQ、o1 等）→ 優先 **Box**
- **多選項資料集**（MMLU-Pro 等）→ **Logit** 最可靠
- **最嚴謹的評測** → Pattern + Logit 同時跑，差異 > 5% 時深入分析

---

## 下一步

若需要評測數學開放題（GSM8K、AIME、MATH 等），請繼續閱讀：

👉 **`02_數學評測.ipynb`** — 數學評測原理、為何不能用 Logit、MathRuler 語意等價判斷詳解